# E-com customer churn analysis
* I got this dataset from Kaggle, and it contains information such as customers' personal details, satisfaction scores, preferred payment mode, days since the last order, and cashback amount. 

* I used python , SQL to clean and analyze this dataset, and performed visualizations using Microsoft Power BI. 

* This analysis is divided into several stages: data cleaning, data exploration, an insight section, interactive dashboard.

In [1]:
from sqlalchemy import create_engine, text
import sqlalchemy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)

Total Tables: 1
Table Names:
- ecommerce_churn


In [ ]:
for table in tables:
       print(f"\n Table: {table}")
       query = text(f"SELECT COUNT(*) FROM {table}")
       df = pd.read_sql_query(query, engine)
       print(f"{table}", df.iloc[0,0])
       display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))


 Table: ecommerce_churn
ecommerce_churn 5630


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130


In [6]:
df= pd.read_sql(f"SELECT * FROM {tables[0]}", engine)
df

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5625,55626,0,10.0,Computer,1,30.0,Credit Card,Male,3.0,2,Laptop & Accessory,1,Married,6,0,18.0,1.0,2.0,4.0,151
5626,55627,0,13.0,Mobile Phone,1,13.0,Credit Card,Male,3.0,5,Fashion,5,Married,6,0,16.0,1.0,2.0,NaN,225
5627,55628,0,1.0,Mobile Phone,1,11.0,Debit Card,Male,3.0,2,Laptop & Accessory,4,Married,3,1,21.0,1.0,2.0,4.0,186
5628,55629,0,23.0,Computer,3,9.0,Credit Card,Male,4.0,5,Laptop & Accessory,4,Married,4,0,15.0,2.0,2.0,9.0,179


In [9]:
df = df.replace("NaN", np.nan)

In [12]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns

Index(['customerid', 'churn', 'tenure', 'preferredlogindevice', 'citytier',
       'warehousetohome', 'preferredpaymentmode', 'gender', 'hourspendonapp',
       'numberofdeviceregistered', 'preferedordercat', 'satisfactionscore',
       'maritalstatus', 'numberofaddress', 'complain',
       'orderamounthikefromlastyear', 'couponused', 'ordercount',
       'daysincelastorder', 'cashbackamount'],
      dtype='object')

In [16]:
df =df.rename(columns = {
    'tenure': 'tenure(no_of_year_stay)',
    'hourspendonapp' : "hour_spend_on_app",
    "numberofdeviceregistered" : "number_of_device_registered",
    "preferedordercat"  : "order_cat",
    "orderamounthikefromlastyear" : "order_amount_hike_from_last_year",
    "daysincelastorder" : "days_since_last_order",
    "numberofaddress" : "number_of_address",
    "preferredpaymentmode" : "payment_mode",
    "preferredlogindevice" : "login_device",
    "maritalstatus" : "marital_status",
    "daysincelastorder" : "days_since_last_order",
    "warehousetohome" : "warehouse_to_home",
})

In [17]:
df.info()
print("--"* 20)
print(f"null chheck : \n{df.isnull().sum()}")
print("--"* 20)
print(f"duplicate check : \n{df.duplicated().sum()}")
print("--"* 20)
print(f" data shape : {df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   customerid                        5630 non-null   int64  
 1   churn                             5630 non-null   int64  
 2   tenure(no_of_year_stay)           5366 non-null   float64
 3   login_device                      5630 non-null   object 
 4   citytier                          5630 non-null   int64  
 5   warehouse_to_home                 5379 non-null   float64
 6   payment_mode                      5630 non-null   object 
 7   gender                            5630 non-null   object 
 8   hour_spend_on_app                 5375 non-null   float64
 9   number_of_device_registered       5630 non-null   int64  
 10  order_cat                         5630 non-null   object 
 11  satisfactionscore                 5630 non-null   int64  
 12  marita

In [20]:
# handle null values with their median 
num_cols = [
    "tenure(no_of_year_stay)",
    "warehouse_to_home",
    "hour_spend_on_app",
    "order_amount_hike_from_last_year",
    "couponused" ,
    "ordercount",
    "days_since_last_order"
]

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [22]:
print("after handling null values")
print("--"* 20)
print(df.isnull().sum())

after handling null values
----------------------------------------
customerid                          0
churn                               0
tenure(no_of_year_stay)             0
login_device                        0
citytier                            0
warehouse_to_home                   0
payment_mode                        0
gender                              0
hour_spend_on_app                   0
number_of_device_registered         0
order_cat                           0
satisfactionscore                   0
marital_status                      0
number_of_address                   0
complain                            0
order_amount_hike_from_last_year    0
couponused                          0
ordercount                          0
days_since_last_order               0
cashbackamount                      0
dtype: int64


In [23]:
# remove white space from object 
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [24]:
for col in df.select_dtypes(include='object').columns:
    if df[col].str.startswith(' ', na=False).any() or df[col].str.endswith(' ', na=False).any():
        print(f"Whitespace still exists in: {col}")

In [25]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    print(f"\n{col} ({len(vals)} unique): {sorted(vals)}")


login_device (3 unique): ['Computer', 'Mobile Phone', 'Phone']

payment_mode (7 unique): ['CC', 'COD', 'Cash on Delivery', 'Credit Card', 'Debit Card', 'E wallet', 'UPI']

gender (2 unique): ['Female', 'Male']

order_cat (6 unique): ['Fashion', 'Grocery', 'Laptop & Accessory', 'Mobile', 'Mobile Phone', 'Others']

marital_status (3 unique): ['Divorced', 'Married', 'Single']


**here in payment method CC means credit card and COD means Cash on delivery so i replace both (CC -> credit card , COD-> cash on delivery)**

In [29]:
df["payment_mode"] = df["payment_mode"].replace({
    "CC": "Credit Card",
    "COD": "Cash on Delivery"
})

print(sorted(df["payment_mode"].unique()))

['Cash on Delivery', 'Credit Card', 'Debit Card', 'E wallet', 'UPI']


In [30]:
df.to_sql(name="clean_dataset",
          con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print("clean dataset save in mysql")

clean dataset save in mysql


In [31]:
df_verify = pd.read_sql("SELECT * FROM clean_dataset LIMIT 5", engine)
display(df_verify)

,customerid,churn,tenure(no_of_year_stay),login_device,citytier,warehouse_to_home,payment_mode,gender,hour_spend_on_app,number_of_device_registered,order_cat,satisfactionscore,marital_status,number_of_address,complain,order_amount_hike_from_last_year,couponused,ordercount,days_since_last_order,cashbackamount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,9.0,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,9.0,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,Credit Card,Male,3.0,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130


In [33]:
df['customerid'].nunique()

5630

In [32]:
original_count = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_dataset", engine)
print(f"✓ Rows in MySQL  : {original_count['cnt'][0]}")
print(f"✓ Rows in df     : {len(df)}")
print(f"✓ Match          : {original_count['cnt'][0] == len(df)}")

✓ Rows in MySQL  : 5630
✓ Rows in df     : 5630
✓ Match          : True


In [34]:
df.to_csv('clean_data.csv',index=False)